# Notebook 01: Data Understanding & Validation

## Credit Card Customer Churn & Segmentation
**Domain:** Banking & Financial Services  
**Objective:** Ingest the raw credit-card cardholder dataset, inspect schema integrity, validate data quality, analyze target variable imbalance, and build a comprehensive data dictionary.

---
### Business Context
Customer attrition directly deprives credit card issuers of interest margins, interchange swipe fees, and annual card fees, while customer acquisition costs (CAC) in retail banking typically exceed annual retention costs by 5x to 7x. 
This project focuses on identifying early attrition indicators and discovering natural behavioral segments to guide proactive retention.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from src.data.load_data import load_raw_data, validate_data, generate_data_dictionary
from src.utils.helpers import RAW_DATA_FILE, TARGET_COLUMN, ID_COLUMN

# Load raw dataset
df = load_raw_data(RAW_DATA_FILE)
print(f"Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head(5)

### Data Quality Checks
We verify:
1. Absence of null values
2. Absence of duplicate rows or duplicate account numbers (`CLIENTNUM`)
3. Target class balance (`Attrition_Flag`)
4. Representation of 'Unknown' categories in demographic columns

In [ ]:
validation = validate_data(df)
print("Is dataset strictly valid?", validation["is_valid"])
print(f"Total nulls: {validation['total_nulls']}")
print(f"Duplicate rows: {validation['duplicate_rows']}")
print(f"Duplicate customer IDs: {validation['duplicate_ids']}")
print("\nTarget distribution:")
for k, v in validation["target_distribution"].items():
    print(f" - {k}: {v:,} ({v / len(df):.2%})")

### Inspecting 'Unknown' Categories
Rather than automatically dropping or imputing `'Unknown'` strings, we analyze their representation across categorical columns.

In [ ]:
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols:
    unknown_ct = (df[col] == 'Unknown').sum()
    if unknown_ct > 0:
        print(f"Column '{col}': {unknown_ct:,} Unknown records ({unknown_ct / len(df):.2%})")

### Data Dictionary
We construct a machine-readable data dictionary classifying each feature into its domain group, data type, and predictive role.

In [ ]:
data_dict = generate_data_dictionary(df)
dict_df = pd.DataFrame(data_dict)[['column_name', 'data_type', 'feature_group', 'used_for_modeling', 'has_unknown', 'description']]
dict_df

### Key Takeaways from Data Understanding
- The dataset consists of **10,127 account records** across **21 features**.
- Target variable `Attrition_Flag` shows a **16.07% churn rate** (1,627 attrited vs 8,500 retained customers), creating a notable class imbalance.
- No null values or duplicate accounts exist.
- `CLIENTNUM` acts strictly as an identifier and must be excluded from predictive modeling.
- Next step: In **Notebook 02**, we perform comprehensive Exploratory Data Analysis (EDA).